# Case Study 1: Iceland vs Eurozone Volatility Comparison

**Objective**: Compare capital flow volatility between Iceland and the Eurozone (1999-2024)

**Methodology**: F-test for equality of variances across 14 capital flow indicators

**Expected Finding**: Iceland shows significantly higher volatility in most indicators

This notebook extracts and transparently shows all statistical calculations from the dashboard implementation.

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add the lib directory to path for stats_core
sys.path.append('../lib')
from stats_core import calculate_f_statistic, get_significance_stars

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

print("Setup complete. Libraries loaded.")

Setup complete. Libraries loaded.


## 2. Load and Explore Data

In [3]:
# Load the comprehensive dataset
data_path = '../data/Clean/comprehensive_df_PGDP_labeled.csv'
df = pd.read_csv(data_path)

In [4]:
df.head()

,COUNTRY,YEAR,QUARTER,UNIT,"Gross domestic product (GDP), Current prices, US dollar","Assets - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank_PGDP","Liabilities - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank_PGDP","Assets - Other investment, Debt instruments_PGDP","Liabilities - Portfolio investment, Equity and investment fund shares_PGDP","Liabilities - Portfolio investment, Total financial assets/liabilities_PGDP","Assets - Portfolio investment, Equity and investment fund shares_PGDP","Liabilities - Portfolio investment, Debt securities_PGDP","Net (net acquisition of financial assets less net incurrence of liabilities) - Direct investment, Total financial assets/liabilities_PGDP","Net (net acquisition of financial assets less net incurrence of liabilities) - Portfolio investment, Total financial assets/liabilities_PGDP","Liabilities - Direct investment, Total financial assets/liabilities_PGDP","Assets - Portfolio investment, Debt securities_PGDP","Assets - Direct investment, Total financial assets/liabilities_PGDP","Assets - Portfolio investment, Total financial assets/liabilities_PGDP","Net (net acquisition of financial assets less net incurrence of liabilities) - Other investment, Total financial assets/liabilities_PGDP",Net (net acquisition of financial assets less net incurrence of liabilities) - Financial derivatives (other than reserves) and employee stock options_PGDP,"Net (net acquisition of financial assets less net incurrence of liabilities) - Financial account balance, excluding reserves and related items_PGDP",CS1_GROUP,CS2_GROUP,CS3_GROUP
0,Portugal,1999,1,"US dollar, Nominal (Current Prices), % of GDP",127597000000.000000,-6.054750,0.105483,-4.138467,-1.940895,0.042193,4.198241,1.983089,-6.184846,17.014480,0.949351,12.858432,-5.235495,17.060190,-17.872412,0.021097,-7.011133,Eurozone,NaN,NaN
1,Portugal,1999,2,"US dollar, Nominal (Current Prices), % of GDP",127597000000.000000,-3.936180,9.147973,0.115965,-0.944286,7.650370,2.425323,8.594655,3.608165,-6.633192,-0.417474,-1.408145,3.190691,1.017178,-4.850646,-0.205424,-8.077783,Eurozone,NaN,NaN
2,Portugal,1999,3,"US dollar, Nominal (Current Prices), % of GDP",127597000000.000000,7.179070,14.982694,7.251387,3.017576,14.384438,0.808632,11.366861,6.955546,-11.675851,-0.279405,1.899955,6.672854,2.708587,-1.028869,-0.088752,-5.841212,Eurozone,NaN,NaN
3,Portugal,1999,4,"US dollar, Nominal (Current Prices), % of GDP",127597000000.000000,-0.273334,-2.290802,-1.041274,2.521835,8.802017,-0.820003,6.280182,1.604212,-8.479873,2.069532,1.142147,3.670490,0.322144,-0.016270,-0.344922,-7.227091,Eurozone,NaN,NaN
4,Portugal,2000,1,"US dollar, Nominal (Current Prices), % of GDP",118658000000.000000,11.967981,15.692388,15.336574,1.207107,9.567069,0.618518,8.359963,4.482589,-6.750487,9.503887,2.194740,13.986477,2.816582,-10.960396,-0.552010,-13.767003,Eurozone,NaN,NaN


In [5]:
print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nDate range: {df['YEAR'].min()} to {df['YEAR'].max()}")
print(f"\nCS1 Groups available:")
print(df['CS1_GROUP'].value_counts())

Data loaded: 2901 rows, 24 columns

Date range: 1999 to 2025

CS1 Groups available:
CS1_GROUP
Eurozone    988
Iceland     105
Name: count, dtype: int64


## 3. Filter Iceland and Eurozone Data

In [9]:
# Filter for Iceland and Eurozone
iceland_data = df[df['CS1_GROUP'] == 'Iceland'].copy()
eurozone_data = df[df['CS1_GROUP'] == 'Eurozone'].copy()

print(f"Iceland data: {iceland_data.shape[0]} observations")
print(f"Eurozone data: {eurozone_data.shape[0]} observations")

# Check date alignment
iceland_dates = set(iceland_data['YEAR'])
eurozone_dates = set(eurozone_data['YEAR'])
common_dates = iceland_dates.intersection(eurozone_dates)
print(f"\nCommon dates: {len(common_dates)} years")

Iceland data: 105 observations
Eurozone data: 988 observations

Common dates: 27 years


## 4. Identify Capital Flow Indicators

Extract the 14 indicators used in CS1 analysis

In [10]:
# Define the 14 indicators from the baseline
# Using actual column names from the data
indicators = [
    'Assets - Direct investment, Total financial assets/liabilities_PGDP',
    'Liabilities - Direct investment, Total financial assets/liabilities_PGDP',
    'Net (net acquisition of financial assets less net incurrence of liabilities) - Direct investment, Total financial assets/liabilities_PGDP',
    'Assets - Portfolio investment, Total financial assets/liabilities_PGDP',
    'Liabilities - Portfolio investment, Total financial assets/liabilities_PGDP',
    'Net (net acquisition of financial assets less net incurrence of liabilities) - Portfolio investment, Total financial assets/liabilities_PGDP',
    'Assets - Portfolio investment, Debt securities_PGDP',
    'Liabilities - Portfolio investment, Debt securities_PGDP',
    'Assets - Portfolio investment, Equity and investment fund shares_PGDP',
    'Liabilities - Portfolio investment, Equity and investment fund shares_PGDP',
    'Net (net acquisition of financial assets less net incurrence of liabilities) - Other investment, Total financial assets/liabilities_PGDP',
    'Assets - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank_PGDP',
    'Assets - Other investment, Debt instruments_PGDP',
    'Liabilities - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank_PGDP'
]

# Check which indicators are available in the data
available_indicators = [col for col in indicators if col in df.columns]
print(f"Found {len(available_indicators)} of {len(indicators)} indicators")

# If not all found, list available columns with PGDP
if len(available_indicators) < len(indicators):
    missing = [ind for ind in indicators if ind not in df.columns]
    print(f"\nMissing indicators:")
    for ind in missing:
        print(f"  - {ind}")
    
    pgdp_cols = [col for col in df.columns if 'PGDP' in col]
    print(f"\nAvailable PGDP columns: {len(pgdp_cols)}")
    for col in pgdp_cols:
        print(f"  - {col}")

Found 14 of 14 indicators


## 5. Transparent F-Test Calculations

For each indicator, we will:
1. Extract Iceland and Eurozone values
2. Show descriptive statistics
3. Calculate F-statistic step by step
4. Determine significance

In [11]:
# Initialize results list
results = []

# Process each indicator
for i, indicator in enumerate(available_indicators[:5], 1):  # Start with first 5
    print(f"\n{'='*70}")
    print(f"INDICATOR {i}: {indicator}")
    print(f"{'='*70}")
    
    # Extract values
    iceland_vals = iceland_data[indicator].dropna()
    eurozone_vals = eurozone_data[indicator].dropna()
    
    # Show data summary
    print(f"\nData Summary:")
    print(f"  Iceland:  n={len(iceland_vals)}, mean={iceland_vals.mean():.6f}, var={iceland_vals.var():.6f}")
    print(f"  Eurozone: n={len(eurozone_vals)}, mean={eurozone_vals.mean():.6f}, var={eurozone_vals.var():.6f}")
    
    # Calculate F-test
    result = calculate_f_statistic(iceland_vals, eurozone_vals, "Iceland", "Eurozone")
    
    # Show calculation details
    print(f"\nF-Test Calculation:")
    print(f"  F = var(Iceland) / var(Eurozone) = {result['var1']:.6f} / {result['var2']:.6f} = {result['f_statistic']:.6f}")
    print(f"  Degrees of freedom: df1={result['n1']-1}, df2={result['n2']-1}")
    print(f"  P-value (two-tailed): {result['p_value']:.8f}")
    
    # Determine significance
    sig_stars = get_significance_stars(result['p_value'])
    print(f"\nSignificance:")
    print(f"  5% level: {'YES' if result['p_value'] < 0.05 else 'NO'}")
    print(f"  1% level: {'YES' if result['p_value'] < 0.01 else 'NO'}")
    print(f"  Stars: {sig_stars if sig_stars else 'Not significant'}")
    
    # Interpretation
    iceland_higher = result['var1'] > result['var2']
    print(f"\nInterpretation:")
    print(f"  Iceland has {'HIGHER' if iceland_higher else 'LOWER'} volatility than Eurozone")
    
    # Store results
    results.append({
        'Indicator': indicator,
        'F_Statistic': result['f_statistic'],
        'P_Value': result['p_value'],
        'Iceland_Var': result['var1'],
        'Eurozone_Var': result['var2'],
        'Iceland_Higher_Volatility': iceland_higher,
        'Significant_5pct': result['p_value'] < 0.05,
        'Significant_1pct': result['p_value'] < 0.01,
        'Significance': sig_stars
    })


INDICATOR 1: Assets - Direct investment, Total financial assets/liabilities_PGDP

Data Summary:
  Iceland:  n=105, mean=3.104997, var=531.114144
  Eurozone: n=988, mean=6.314956, var=346.378569

F-Test Calculation:
  F = var(Iceland) / var(Eurozone) = 531.114144 / 346.378569 = 1.533334
  Degrees of freedom: df1=104, df2=987
  P-value (two-tailed): 0.00167270

Significance:
  5% level: YES
  1% level: YES
  Stars: ***

Interpretation:
  Iceland has HIGHER volatility than Eurozone

INDICATOR 2: Liabilities - Direct investment, Total financial assets/liabilities_PGDP

Data Summary:
  Iceland:  n=105, mean=3.938255, var=246.754308
  Eurozone: n=988, mean=5.611882, var=364.778675

F-Test Calculation:
  F = var(Iceland) / var(Eurozone) = 246.754308 / 364.778675 = 0.676449
  Degrees of freedom: df1=104, df2=987
  P-value (two-tailed): 0.01231490

Significance:
  5% level: YES
  1% level: NO
  Stars: **

Interpretation:
  Iceland has LOWER volatility than Eurozone

INDICATOR 3: Net (net acqui

## 6. Process All Indicators

In [12]:
# Process remaining indicators (less verbose)
all_results = []

for indicator in available_indicators:
    # Extract values
    iceland_vals = iceland_data[indicator].dropna()
    eurozone_vals = eurozone_data[indicator].dropna()
    
    # Calculate F-test
    result = calculate_f_statistic(iceland_vals, eurozone_vals, "Iceland", "Eurozone")
    
    # Store results
    all_results.append({
        'Indicator': indicator,
        'F_Statistic': result['f_statistic'],
        'P_Value': result['p_value'],
        'Iceland_Higher_Volatility': result['var1'] > result['var2'],
        'Significant_5pct': result['p_value'] < 0.05,
        'Significant_1pct': result['p_value'] < 0.01
    })

# Create results dataframe
results_df = pd.DataFrame(all_results)
print(f"Processed {len(results_df)} indicators")
results_df

Processed 14 indicators


,Indicator,F_Statistic,P_Value,Iceland_Higher_Volatility,Significant_5pct,Significant_1pct
0,"Assets - Direct investment, Total financial as...",1.533334,0.001673,True,True,True
1,"Liabilities - Direct investment, Total financi...",0.676449,0.012315,False,True,False
2,Net (net acquisition of financial assets less ...,3.387842,0.000000,True,True,True
3,"Assets - Portfolio investment, Total financial...",0.624487,0.002784,False,True,True
4,"Liabilities - Portfolio investment, Total fina...",2.064968,0.000000,True,True,True
5,Net (net acquisition of financial assets less ...,3.813782,0.000000,True,True,True
6,"Assets - Portfolio investment, Debt securities...",0.482604,0.000007,False,True,True
7,"Liabilities - Portfolio investment, Debt secur...",7.925142,0.000000,True,True,True
8,"Assets - Portfolio investment, Equity and inve...",1.453813,0.006205,True,True,True
9,"Liabilities - Portfolio investment, Equity and...",0.053866,0.000000,False,True,True


## 7. Summary Statistics

In [13]:
# Summary of findings
print("="*70)
print("SUMMARY OF FINDINGS")
print("="*70)

# Count significant results
sig_5pct = results_df['Significant_5pct'].sum()
sig_1pct = results_df['Significant_1pct'].sum()
iceland_higher = results_df['Iceland_Higher_Volatility'].sum()

print(f"\nStatistical Significance:")
print(f"  Significant at 5% level: {sig_5pct}/{len(results_df)} indicators")
print(f"  Significant at 1% level: {sig_1pct}/{len(results_df)} indicators")

print(f"\nVolatility Comparison:")
print(f"  Iceland has higher volatility: {iceland_higher}/{len(results_df)} indicators")
print(f"  Eurozone has higher volatility: {len(results_df) - iceland_higher}/{len(results_df)} indicators")

# Significant and Iceland higher
sig_and_higher = results_df[results_df['Significant_5pct'] & results_df['Iceland_Higher_Volatility']]
print(f"\nIceland significantly higher volatility (5% level): {len(sig_and_higher)} indicators")

# Show indicators where Iceland has significantly higher volatility
if len(sig_and_higher) > 0:
    print("\nIndicators with significantly higher Iceland volatility:")
    for idx, row in sig_and_higher.iterrows():
        print(f"  - {row['Indicator']}: F={row['F_Statistic']:.3f}, p={row['P_Value']:.6f}")

SUMMARY OF FINDINGS

Statistical Significance:
  Significant at 5% level: 14/14 indicators
  Significant at 1% level: 13/14 indicators

Volatility Comparison:
  Iceland has higher volatility: 10/14 indicators
  Eurozone has higher volatility: 4/14 indicators

Iceland significantly higher volatility (5% level): 10 indicators

Indicators with significantly higher Iceland volatility:
  - Assets - Direct investment, Total financial assets/liabilities_PGDP: F=1.533, p=0.001673
  - Net (net acquisition of financial assets less net incurrence of liabilities) - Direct investment, Total financial assets/liabilities_PGDP: F=3.388, p=0.000000
  - Liabilities - Portfolio investment, Total financial assets/liabilities_PGDP: F=2.065, p=0.000000
  - Net (net acquisition of financial assets less net incurrence of liabilities) - Portfolio investment, Total financial assets/liabilities_PGDP: F=3.814, p=0.000000
  - Liabilities - Portfolio investment, Debt securities_PGDP: F=7.925, p=0.000000
  - Assets 

## 8. Verify Against Baseline

Compare our results with the dashboard baseline to ensure accuracy

In [14]:
# Load baseline results
baseline_df = pd.read_csv('../verification/baseline_results/CS1_baseline.csv')
print(f"Baseline has {len(baseline_df)} indicators")

# Map our indicator names to baseline names (they should match now)
indicator_mapping = {
    'Assets - Direct investment, Total financial assets/liabilities_PGDP': 'Assets - Direct investment, Total financial assets/liabilities',
    'Liabilities - Direct investment, Total financial assets/liabilities_PGDP': 'Liabilities - Direct investment, Total financial assets/liabilities',
    'Net (net acquisition of financial assets less net incurrence of liabilities) - Direct investment, Total financial assets/liabilities_PGDP': 'Net - Direct investment, Total financial assets/liabilities',
    'Assets - Portfolio investment, Total financial assets/liabilities_PGDP': 'Assets - Portfolio investment, Total financial assets/liabilities',
    'Liabilities - Portfolio investment, Total financial assets/liabilities_PGDP': 'Liabilities - Portfolio investment, Total financial assets/liabilities',
    'Net (net acquisition of financial assets less net incurrence of liabilities) - Portfolio investment, Total financial assets/liabilities_PGDP': 'Net - Portfolio investment, Total financial assets/liabilities',
    'Assets - Portfolio investment, Debt securities_PGDP': 'Assets - Portfolio investment, Debt securities',
    'Liabilities - Portfolio investment, Debt securities_PGDP': 'Liabilities - Portfolio investment, Debt securities',
    'Assets - Portfolio investment, Equity and investment fund shares_PGDP': 'Assets - Portfolio investment, Equity and investment fund shares',
    'Liabilities - Portfolio investment, Equity and investment fund shares_PGDP': 'Liabilities - Portfolio investment, Equity and investment fund shares',
    'Net (net acquisition of financial assets less net incurrence of liabilities) - Other investment, Total financial assets/liabilities_PGDP': 'Net - Other investment, Total financial assets/liabilities',
    'Assets - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank_PGDP': 'Assets - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank',
    'Assets - Other investment, Debt instruments_PGDP': 'Assets - Other investment, Debt instruments',
    'Liabilities - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank_PGDP': 'Liabilities - Other investment, Debt instruments, Deposit taking corporations, except the Central Bank'
}

# For each of our results, find and compare with baseline
verification_results = []
tolerance = 0.0001

for _, row in results_df.iterrows():
    indicator = row['Indicator']
    baseline_name = indicator_mapping.get(indicator)
    
    if baseline_name:
        baseline_row = baseline_df[baseline_df['Indicator'] == baseline_name]
        if not baseline_row.empty:
            baseline_f = baseline_row['F_Statistic'].values[0]
            baseline_p = baseline_row['P_Value'].values[0]
            
            f_diff = abs(row['F_Statistic'] - baseline_f)
            p_diff = abs(row['P_Value'] - baseline_p)
            
            verification_results.append({
                'Indicator': indicator.replace('_PGDP', ''),  # Simplify name for display
                'Our_F': row['F_Statistic'],
                'Baseline_F': baseline_f,
                'F_Diff': f_diff,
                'F_Match': f_diff < tolerance,
                'Our_P': row['P_Value'],
                'Baseline_P': baseline_p,
                'P_Diff': p_diff,
                'P_Match': p_diff < 0.01  # More lenient for p-values due to numerical precision
            })

verification_df = pd.DataFrame(verification_results)

# Show verification summary
if len(verification_df) > 0:
    print(f"\nVerification Results:")
    print(f"  Indicators matched: {len(verification_df)}/{len(results_df)}")
    print(f"  F-statistics within tolerance: {verification_df['F_Match'].sum()}/{len(verification_df)}")
    print(f"  P-values reasonably close: {verification_df['P_Match'].sum()}/{len(verification_df)}")
    
    # Show any discrepancies
    discrepancies = verification_df[~verification_df['F_Match']]
    if len(discrepancies) > 0:
        print(f"\nF-statistic discrepancies found in {len(discrepancies)} indicators:")
        for _, row in discrepancies.iterrows():
            print(f"  {row['Indicator'][:50]}...")
            print(f"    F-stat: {row['Our_F']:.6f} vs {row['Baseline_F']:.6f} (diff: {row['F_Diff']:.8f})")
    else:
        print("\nAll F-statistics match baseline within tolerance!")
else:
    print("Could not verify - check indicator mapping")

Baseline has 14 indicators

Verification Results:
  Indicators matched: 14/14
  F-statistics within tolerance: 14/14
  P-values reasonably close: 14/14

All F-statistics match baseline within tolerance!


## 9. Save Results

In [15]:
# Save results to CSV
output_path = '../outputs/CS1_results.csv'
results_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

# Save verification results if available
if len(verification_df) > 0:
    verification_path = '../outputs/CS1_verification.csv'
    verification_df.to_csv(verification_path, index=False)
    print(f"Verification saved to: {verification_path}")

# Create summary report
summary = {
    'Total_Indicators': len(results_df),
    'Significant_5pct': sig_5pct,
    'Significant_1pct': sig_1pct,
    'Iceland_Higher_Count': iceland_higher,
    'Iceland_Significantly_Higher': len(sig_and_higher),
    'Verification_Pass': verification_df['F_Match'].all() if len(verification_df) > 0 else False
}

summary_df = pd.DataFrame([summary])
summary_path = '../outputs/CS1_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)

Results saved to: ../outputs/CS1_results.csv
Verification saved to: ../outputs/CS1_verification.csv
Summary saved to: ../outputs/CS1_summary.csv

ANALYSIS COMPLETE


## 10. Key Findings

Based on the F-test analysis comparing Iceland and Eurozone capital flow volatility (1999-2024):

1. **Iceland shows significantly higher volatility** in the majority of capital flow indicators
2. **Statistical significance** is strong, with most differences significant at the 1% level
3. **Portfolio debt securities** show the largest volatility difference (F-statistic > 7)
4. **Results match the dashboard baseline**, confirming calculation accuracy

This supports the hypothesis that small open economies like Iceland experience higher capital flow volatility compared to larger currency unions like the Eurozone.